# Support Vector Machine (SVM)

The Support Vector Machine is notoriously slow on data with high dimensionality. We try to use PCA decomposition, which chooses the features with the highest variance, to solve this issue.

In [2]:
# Data Handling and Analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

###
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

###
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [3]:
# Read csv file and look at contents
file_path = "T49.2_Sep2025_1_StGallen.csv"
df = pd.read_csv(file_path, sep=";", na_values=["", " ", "NA", "N/A", "nan", "NaN"])

print(df.shape)
print(df.head())

# Proportions of classes
print("Value Counts (in %):")
class_shares = df["OUTCOME_3Kat_KHK"].value_counts(normalize=True) * 100
print(class_shares.round(2))
print("Number of nan")
print(df.isnull().sum())
df.isnull().sum().sort_values(ascending=False).head(20)

# Gesamtanzahl leerer Zellen im gesamten DataFrame
print("Gesamtanzahl Zellen:", df.size)
print("Gesamtanzahl leerer Zellen:", df.isnull().sum().sum())

(2310, 649)
   OUTCOME_3Kat_KHK           Alter_0  geschlecht waist_0             WHR_0  \
0                 2  56,9691991786448           0     112  ,982456140350877   
1                 2  69,4976043805613           0     104              1,04   
2                 2  72,0492813141684           0     129  ,941605839416058   
3                 2  70,8145106091718           0      99  ,876106194690266   
4                 2  62,1601642710472           1     125               NaN   

              BMI_0  currsmo0  RR_syst_0  RR_diast_0  Pulse_Pressure  ...  \
0  32,8731097961867         0      130.0        80.0            50.0  ...   
1  30,1102788964583         0      160.0        70.0            90.0  ...   
2  44,7348800491207         0      140.0        90.0            50.0  ...   
3  26,2595847484332         0      130.0        90.0            40.0  ...   
4  42,4366343891054         1      180.0       100.0            80.0  ...   

   SMC231l  SMC232l  SMC233l  SMC240l  SMC241l SMC

/var/folders/rp/sz_1shj9783116jtrj1kc04m0000gn/T/ipykernel_27572/2271635866.py:3: DtypeWarning: Columns (3,32,119,121,125,129,231,330,342,588) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=";", na_values=["", " ", "NA", "N/A", "nan", "NaN"])


In [4]:
# SVM comparison: LinearSVC vs. SVC (RBF) with PCA

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n=== {name} ===")
    print(f"Train time: {train_time:.2f} s")
    print(f"Accuracy:   {acc:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    return acc

# 1) Linear SVM (fast, good for many features)
linear_svm = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),   # works with sparse One-Hot; safe for dense too
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",   # handle imbalance
        max_iter=10000,
        dual="auto",
        random_state=42
    ))
])

# 2) RBF SVM with PCA (reduce 649 dims -> ~100; speeds up and helps generalization)
n_pca = min(100, X_train.shape[1], max(10, X_train.shape[0] - 1))  # cap at 100 comps
svm_pca_rbf = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("pca", PCA(n_components=n_pca, random_state=42)),
    ("svm", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",   # handle imbalance
        probability=False,         # keep False for speed
        random_state=42
    ))
])

acc_lin = evaluate_model("LinearSVC (scaled)", linear_svm, X_train, y_train, X_test, y_test)
acc_rbf = evaluate_model(f"SVC RBF + PCA({n_pca})", svm_pca_rbf, X_train, y_train, X_test, y_test)

print("\nSummary:")
print(f"- LinearSVC Accuracy:     {acc_lin:.4f}")
print(f"- SVC RBF + PCA Accuracy: {acc_rbf:.4f}")


NameError: name 'X_train' is not defined

In [5]:


def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print("\n" + "="*78)
    print(f"=== {name} ===")
    print(f"Train time: {train_time:.2f} s")
    print(f"Accuracy:   {acc:.4f} | Macro-F1: {macro_f1:.4f} | Balanced Acc: {bal_acc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    # digits=3 prevents Jupyter from collapsing; printing the string avoids truncation artifacts
    report_str = classification_report(y_test, y_pred, digits=3, zero_division=0)
    print(report_str)
    print("="*78)
    return {"model": name, "accuracy": acc, "macro_f1": macro_f1, "balanced_acc": bal_acc, "train_time_s": train_time}

# 1) Linear SVM (fast, good for many features)
linear_svm = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),   # safe for one-hot / sparse-like design
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=10000,
        dual="auto",
        random_state=42
    ))
])

# 2) RBF SVM with PCA (reduce 649 -> ~100 comps to speed up & improve generalization)
n_pca = min(100, X_train.shape[1], max(10, X_train.shape[0] - 1))  # cap at 100 comps
svm_pca_rbf = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("pca", PCA(n_components=n_pca, random_state=42)),
    ("svm", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        probability=False,    # keep False for speed
        random_state=42
    ))
])

# Run & collect metrics
res_lin = evaluate_model("LinearSVC (scaled)", linear_svm, X_train, y_train, X_test, y_test)
res_rbf = evaluate_model(f"SVC RBF + PCA({n_pca})", svm_pca_rbf, X_train, y_train, X_test, y_test)

# Compact summary
summary = pd.DataFrame([res_lin, res_rbf]).sort_values("macro_f1", ascending=False)
print("\nSummary (sorted by Macro-F1):")
print(summary.to_string(index=False))


NameError: name 'X_train' is not defined